<a href="https://colab.research.google.com/github/MettaTan/SIT-UofG-QC-Assignment/blob/main/BB84_Attacker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 19.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 552.4 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=bf24c72b0a9d68c65c1f46c9ea8e69a0f7d4d7aa9a7082a24018338cd8a6f3ef
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

## Overview

We demonstrate **two different attack styles** by an eavesdropper Eve and show
that BB84 detects both.

| Attack | What Eve does | Expected error rate | Info Eve gains |
|--------|---------------|--------------------:|---------------:|
| **1 — Intercept-resend** | Measures every qubit in a random basis, then forwards a fresh qubit carrying what she saw | ~25% | ~75% bit-agreement with Alice |
| **2 — Random replacement** | Doesn't measure at all — discards Alice's qubit and sends a randomly prepared one of her own | ~50% | none (~50% agreement = random guessing) |

Attack 1 is "stealthy" — lower disruption but Eve learns a lot.
Attack 2 is "loud" — easily detected and Eve learns nothing.
With a 15% threshold on the error rate, both are caught.

Each agent's section below is labelled **ALICE**, **EVE**, or **BOB**.


In [3]:
simulator = BasicSimulator()


## Quantum random bit generator

Measuring $|+\rangle = \tfrac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ yields a uniform random bit.
Batched in chunks of 16 because `BasicSimulator` caps at 24 qubits per circuit.


In [4]:
CHUNK = 16

def quantum_random_bits(n):
    """Return n uniformly random bits, each from measuring |+>."""
    bits = []
    while len(bits) < n:
        k = min(CHUNK, n - len(bits))
        qc = QuantumCircuit(k, k)
        qc.h(range(k))
        qc.measure(range(k), range(k))
        result = simulator.run(qc, shots=1).result()
        bitstring = list(result.get_counts().keys())[0]
        bits.extend(int(b) for b in bitstring[::-1])
    return bits[:n]


## Helper functions


In [5]:
def prepare_qubit(bit, basis):
    """Encode bit into a qubit in the chosen basis (0 = Z, 1 = X)."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

def measure_qubit(qc, basis):
    """Measure a one-qubit circuit in basis 0 (Z) or 1 (X)."""
    qc = qc.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts().keys())[0])

def sift_and_detect(alice_bits, alice_bases, bob_bits, bob_bases, threshold=0.15):
    """Public-discussion phase: keep matching-bases positions, sample-check the rest."""
    matching = [i for i in range(len(alice_bits)) if alice_bases[i] == bob_bases[i]]
    a_sift = [alice_bits[i] for i in matching]
    b_sift = [bob_bits[i]   for i in matching]

    s = len(matching) // 2
    errs = sum(1 for x, y in zip(a_sift[:s], b_sift[:s]) if x != y)
    rate = errs / s if s else 0.0
    return {
        "matching": matching, "a_sift": a_sift, "b_sift": b_sift,
        "sample_size": s, "errors": errs, "error_rate": rate,
        "detected": rate > threshold,
    }


# Attack 1 — Intercept-resend

Eve picks a random basis for each qubit, measures it, and forwards a fresh qubit
carrying the bit she observed (prepared in her own basis).


In [6]:
N = 128

# ----- ALICE -----
alice_bits   = quantum_random_bits(N)
alice_bases  = quantum_random_bits(N)
alice_qubits = [prepare_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]
print(f"Alice prepared {N} qubits.")

# ----- EVE (intercept-resend) -----
eve_bases        = quantum_random_bits(N)
eve_bits         = []
forwarded_qubits = []
for i in range(N):
    bit = measure_qubit(alice_qubits[i], eve_bases[i])
    eve_bits.append(bit)
    forwarded_qubits.append(prepare_qubit(bit, eve_bases[i]))
print(f"Eve intercepted and forwarded {N} qubits.")

# ----- BOB -----
bob_bases = quantum_random_bits(N)
bob_bits  = [measure_qubit(forwarded_qubits[i], bob_bases[i]) for i in range(N)]
print(f"Bob measured {N} qubits.")


Alice prepared 128 qubits.
Eve intercepted and forwarded 128 qubits.
Bob measured 128 qubits.


## Detection — Attack 1


In [7]:
r1 = sift_and_detect(alice_bits, alice_bases, bob_bits, bob_bases)
print(f"Sifted key length: {len(r1['matching'])}")
print(f"Sample size:       {r1['sample_size']}")
print(f"Errors found:      {r1['errors']}")
print(f"Error rate:        {r1['error_rate']:.2%}")
print(f"Detection:         {'EAVESDROPPING DETECTED' if r1['detected'] else 'channel looks clean'}")

# How much did Eve actually learn?
eve_sifted = [eve_bits[i] for i in r1['matching']]
agree = sum(1 for a, e in zip(r1['a_sift'], eve_sifted) if a == e)
print(f"\nEve's bit-agreement with Alice: {agree}/{len(r1['a_sift'])} "
      f"({agree/len(r1['a_sift']):.1%})")


Sifted key length: 66
Sample size:       33
Errors found:      13
Error rate:        39.39%
Detection:         EAVESDROPPING DETECTED

Eve's bit-agreement with Alice: 48/66 (72.7%)


# Attack 2 — Random replacement

Eve doesn't measure Alice's qubits at all — she discards each one and sends a
freshly prepared qubit (random bit, random basis) of her own. She gains **no
information** about Alice's key, but causes ~50% disruption on the sifted bits,
well above the 15% threshold.

This shows that BB84's security doesn't rely on Eve being clever — even a vandal
who only disrupts the channel is caught.


In [8]:
# ----- ALICE (fresh run for attack 2) -----
alice_bits   = quantum_random_bits(N)
alice_bases  = quantum_random_bits(N)
alice_qubits = [prepare_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]
print(f"Alice prepared {N} qubits.")

# ----- EVE (random replacement, no measurement) -----
fake_bits        = quantum_random_bits(N)
fake_bases       = quantum_random_bits(N)
forwarded_qubits = [prepare_qubit(fake_bits[i], fake_bases[i]) for i in range(N)]
# Alice's original qubits are simply discarded -- Eve never learns anything about them.
print(f"Eve replaced all {N} qubits with random ones (no measurement).")

# ----- BOB -----
bob_bases = quantum_random_bits(N)
bob_bits  = [measure_qubit(forwarded_qubits[i], bob_bases[i]) for i in range(N)]
print(f"Bob measured {N} qubits.")


Alice prepared 128 qubits.
Eve replaced all 128 qubits with random ones (no measurement).
Bob measured 128 qubits.


## Detection — Attack 2


In [9]:
r2 = sift_and_detect(alice_bits, alice_bases, bob_bits, bob_bases)
print(f"Sifted key length: {len(r2['matching'])}")
print(f"Sample size:       {r2['sample_size']}")
print(f"Errors found:      {r2['errors']}")
print(f"Error rate:        {r2['error_rate']:.2%}")
print(f"Detection:         {'EAVESDROPPING DETECTED' if r2['detected'] else 'channel looks clean'}")

# Eve's "knowledge" -- her fake bits are uncorrelated with Alice's, so ~50% agreement.
fake_sifted = [fake_bits[i] for i in r2['matching']]
agree = sum(1 for a, e in zip(r2['a_sift'], fake_sifted) if a == e)
print(f"\nEve's bit-agreement with Alice: {agree}/{len(r2['a_sift'])} "
      f"({agree/len(r2['a_sift']):.1%})  -- expected ~50% (random)")


Sifted key length: 58
Sample size:       29
Errors found:      15
Error rate:        51.72%
Detection:         EAVESDROPPING DETECTED

Eve's bit-agreement with Alice: 28/58 (48.3%)  -- expected ~50% (random)


## Repeated trials — comparing both attacks

Each scenario is run many times to confirm detection is reliable, not lucky.
A baseline with no attacker is included to verify the threshold doesn't produce false alarms.


In [10]:
def run_trial(eve_style, N=64, threshold=0.15):
    """Run one full protocol with the given Eve strategy.
    Returns (detected, error_rate, eve_bit_agreement_or_None)."""
    a_bits  = quantum_random_bits(N)
    a_bases = quantum_random_bits(N)
    a_qubits = [prepare_qubit(a_bits[i], a_bases[i]) for i in range(N)]

    if eve_style == "intercept_resend":
        e_bases = quantum_random_bits(N)
        e_bits, fwd = [], []
        for i in range(N):
            b = measure_qubit(a_qubits[i], e_bases[i])
            e_bits.append(b)
            fwd.append(prepare_qubit(b, e_bases[i]))
    elif eve_style == "random_replacement":
        e_bits  = quantum_random_bits(N)   # Eve's own random bits, no measurement
        e_bases = quantum_random_bits(N)
        fwd = [prepare_qubit(e_bits[i], e_bases[i]) for i in range(N)]
    elif eve_style == "no_attack":
        e_bits = None
        fwd = a_qubits
    else:
        raise ValueError(eve_style)

    b_bases = quantum_random_bits(N)
    b_bits  = [measure_qubit(fwd[i], b_bases[i]) for i in range(N)]

    r = sift_and_detect(a_bits, a_bases, b_bits, b_bases, threshold)
    if e_bits is not None and r['a_sift']:
        e_sift = [e_bits[i] for i in r['matching']]
        agreement = sum(1 for a, e in zip(r['a_sift'], e_sift) if a == e) / len(r['a_sift'])
    else:
        agreement = None
    return r['detected'], r['error_rate'], agreement

TRIALS = 10

for style in ("no_attack", "intercept_resend", "random_replacement"):
    detected = 0
    rates, agrees = [], []
    for _ in range(TRIALS):
        d, rate, agr = run_trial(style)
        detected += int(d)
        rates.append(rate)
        if agr is not None:
            agrees.append(agr)
    mean_rate = sum(rates) / len(rates)
    agree_txt = f",  Eve bit-agreement {sum(agrees)/len(agrees):.1%}" if agrees else ""
    label = "alarms" if style == "no_attack" else "detected"
    print(f"{style:20s}  {label} {detected:2d}/{TRIALS},  "
          f"mean error rate {mean_rate:.2%}{agree_txt}")


no_attack             alarms  0/10,  mean error rate 0.00%
intercept_resend      detected  8/10,  mean error rate 24.67%,  Eve bit-agreement 72.8%
random_replacement    detected  9/10,  mean error rate 43.08%,  Eve bit-agreement 54.7%
